In [1]:
# Python 3.10.11
# %pip install -r requirements.txt > /dev/null
from args import *
from utils import *

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import random_split

# TODO(241225) 依赖导入
import pandas as pd


In [3]:
# TODO(241225) 导入label数据
label_df = pd.read_csv(sample_labels_file_path, index_col=0)

# 裁减样本数量，使得 0 - 1 样本数量一致
df_0 = label_df[label_df['label'] == 0]
df_1 = label_df[label_df['label'] == 1]
min_len = min(len(df_0), len(df_1))
new_df = pd.concat([df_0[:min_len], df_1[:min_len]], axis=0)

sample_key_list = label_df.index.to_list()
sample_key_list = new_df.index.to_list()

print(f"label 样本数量: {len(sample_key_list)}")

# TODO(241225) 导入gene数据
# 根据 sample_key_list 为基准, 若模态数据中不存在 sample_key 则填充新数据
gene_array = load_gene_data_by_sample_key(sample_key_list).values
cnv_array = load_cnv_data_by_sample_key(sample_key_list).values

wsi_array = load_wsi_data_by_sample_key(sample_key_list)
report_array = load_report_data_by_sample_key(sample_key_list)

label_array = label_df.values
label_array = new_df.values

_, gene_dim = gene_array.shape
_, cnv_dim = cnv_array.shape
_, wsi_dim = wsi_array.shape
_, report_dim = report_array.shape
_, label_dim = label_array.shape

print(f"""
gene 数据维度:   {gene_dim}
cnv 数据维度:    {cnv_dim}
wsi 数据维度:    {wsi_dim}
report 数据维度: {report_dim}
""")

batch_size = 2

all_dataset = MultiOmicsDataset(gene_array, cnv_array, report_array, wsi_array, label_array)

# # 假设 all_dataset 是一个 Dataset 对象
# train_len = int(len(all_dataset) * 0.8)  # 80% 的数据用作训练集
# test_len = len(all_dataset) - train_len  # 剩余的数据用作验证集

# # 使用 random_split 分割数据集
# train_val_dataset, test_dataset = random_split(all_dataset, [train_len, test_len])

train_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)
val_loader = DataLoader(all_dataset, batch_size=batch_size, shuffle=True, num_workers=3, drop_last=False)

label 样本数量: 152

gene 数据维度:   2340
cnv 数据维度:    2340
wsi 数据维度:    2048
report 数据维度: 768



In [4]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import models  # 如果需要使用预训练模型

# 假设我们使用一个预训练的ResNet作为特征提取器，并添加自定义层
class CustomModel(nn.Module):
    def __init__(self, base_model, num_classes):
        super(CustomModel, self).__init__()
        # 假设base_model期望的输入形状是[batch_size, 3, 224, 224]
        # 我们将第一层替换为一个全连接层
        self.base_model = nn.Sequential(
            nn.Linear(2048, base_model.fc.in_features),
            nn.ReLU(inplace=True)
        )
        self.custom_layer = nn.Sequential(
            # nn.Linear(base_model.fc.in_features, 256),
            nn.Linear(2048, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x = self.base_model(x)
        x = self.custom_layer(x)
        return x

# 加载预训练模型并替换最后一层
base_model = models.resnet50(pretrained=True)
num_classes = 1  # 根据您的任务确定类别数
model = CustomModel(base_model, num_classes).to(device)

# 定义损失函数和优化器
criterion = nn.BCEWithLogitsLoss()  # 适用于二分类问题
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 学习率调度器，可以在训练过程中调整学习率
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min')

/root/miniforge3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/root/miniforge3/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/root/miniforge3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/miniforge3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Res

In [5]:
num_epochs = 3
for _ in range(num_epochs):


    model.train()
    # 导入 batch 数据
    for batch in train_loader:
        gene_tensor = batch["gene_tensor"].to(device)
        cnv_tensor = batch["cnv_tensor"].to(device)
        report_tensor = batch["report_tensor"].to(device)
        wsi_tensor = batch["wsi_tensor"].to(device)
        label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)

        # outputs = model(gene_tensor, cnv_tensor, report_tensor, wsi_tensor)
        optimizer.zero_grad()
        outputs = model(wsi_tensor)  # 假设wsi_tensor是主要的输入特征
        # print((outputs.shape, label_tensor.shape))
        loss = criterion(outputs, label_tensor)
        loss.backward()
        optimizer.step()

    print("train loss: ", loss.item())
    # 更新学习率
    scheduler.step(loss)




    model.eval()  # 设置模型为评估模式
    all_preds = []
    all_labels = []

    with torch.no_grad():  # 不计算梯度，节省内存
        for batch in val_loader:
            # 假设batch["wsi_tensor"]是输入数据，batch["label_tensor"]是标签
            wsi_tensor = batch["wsi_tensor"].to(device)
            label_tensor = torch.squeeze(batch["label_tensor"]).to(torch.float32).to(device).view(-1, 1)

            outputs = model(wsi_tensor)
            preds = torch.argmax(outputs, dim=1)  # 获取预测的类别索引

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label_tensor.cpu().numpy())

    # 计算准确率
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)

    print(f'Validation Accuracy: {accuracy}')

train loss:  0.3481151759624481
Validation Accuracy: 0.5
train loss:  0.5341969728469849
Validation Accuracy: 0.5
train loss:  0.03570759296417236
Validation Accuracy: 0.5


In [6]:
all_preds

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [8]:
all_labels.squeeze()

array([0., 1., 1., 1., 0., 0., 0., 1., 0., 1., 1., 0., 0., 1., 1., 1., 0.,
       1., 1., 1., 1., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 1., 1., 1.,
       0., 0., 0., 1., 0., 0., 0., 1., 1., 0., 1., 1., 1., 0., 0., 0., 1.,
       0., 0., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 1., 1., 1.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0.,
       1., 1., 1., 0., 0., 0., 1., 1., 0., 0., 1., 0., 1., 0., 0., 0., 1.,
       1., 1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 1., 0., 1.,
       1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 0., 1., 1., 0., 1., 0., 0.,
       1., 1., 0., 0., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0., 0., 1.],
      dtype=float32)